# Exploratory Notebook : Vector Database
## Introduction 

Now that we’ve completed the chunking and embedding steps, we can create our vector database. The idea is that this database stores the vectors we’ve created. Thanks to these vectors, we can search our documents based on their meaning, rather than just using keywords.                

To do this, we’ll use pgvector because:             
- it allows us to store all of our reference documents and vectors in a single database. They are linked together by keys, which makes it possible to quickly find matches
- it performs well with up to several million vectors, which is sufficient for our current volume
- it supports SQL queries, allowing us to search for meaning and retrieve documents in a single query

### 01. Docker compose 

So we'll create a Docker Compose file to ensure full reproducibility. It can be found in the `docker-compose.yml` file.

We use pgvector, which adds vector support to PostgreSQL

To do this, we'll use
-  the pgvector image name, which can be found here: [pgvector github](https://github.com/pgvector/pgvector) 
- the PostgreSQL image: [postgreSQL image documentation](https://hub.docker.com/_/postgres)
 
The credentials are in the `.env` file

## 02. PostgreSQL database 

We will now define the structure of our database. We will define three tables: 
- document: one row per PDF 
- chunk: one row per chunk, including its pages and vectors 
- query: one row per question asked, to keep track of the questions

There is also a table that will help us make a connection: 
- chunk_used_by_query: which lists the chunks requested by the queries 

The basic idea is that tables can be linked together using keys: 
- A chunk is linked to a document
- query and chunk are linked together through the chunk_used_by_query table

We will also add constraints to protect the database 
- No duplicates if we reload data into the database
- If a document is deleted, its chunks are deleted along with it, so that there are no chunks without a document

The database has now been created, we can connect our notebook to it.

In [28]:
import os 
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector 
import numpy as np 
import pandas as pd 
from sentence_transformers import SentenceTransformer
from mistralai.client import Mistral

To establish the connection, we'll use this documentation: [osycopg connection classes documentation](https://www.psycopg.org/psycopg3/docs/api/connections.html)

In [12]:
load_dotenv() 

conn = psycopg.connect(autocommit=True)
register_vector(conn)

Now that we have a connection to our database, we'll load our documents and our embeddings into it

In [13]:
df_chunks = pd.read_json("data/processed/chunks_01.json")
embeddings = np.load("data/processed/embeddings_01.npy")


We retrieve the position of each chunk in its document and add it to our database

In [14]:
df_chunks["chunk_index"] = df_chunks.groupby("origin").cumcount()

All right, now we can insert our documents into the database
[pgvector insertion documentation](https://github.com/pgvector/pgvector-python)

Let's start by filling in the table with the names of the documents 

In [15]:
for name in df_chunks["origin"].unique():
    conn.execute(
        "INSERT INTO document (name) VALUES (%s) ON CONFLICT (name) DO NOTHING",
        (str(name),),
    )

print(conn.execute("SELECT id, name FROM document ORDER BY id").fetchall())

[(1, 'avis_ASNR_2026.pdf'), (2, 'avis_expertise_ASNR_2026.pdf'), (3, 'dossier_concertation_calogena_2026.pdf')]


and the chunk table

In [16]:
documents_ids = dict(conn.execute("SELECT name, id FROM document").fetchall())

for i, row in df_chunks.iterrows():
    conn.execute(
        """
        INSERT INTO chunk
            (document_id, chunk_index, starting_page, ending_page, 
            starting_character, ending_character, text, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (document_id, chunk_index) DO NOTHING
        """,
        (documents_ids[row["origin"]],
         int(row["chunk_index"]),
         int(row["page_start"]),
         int(row["page_end"]),
         int(row["start"]),
         int(row["end"]),
         row["text"],
         embeddings[i],
        ),
    )

All right, now we can ask a question. It has to be converted into a vector itself. 

In [25]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

question = "Que doit faire Calogena après l'avis d'expertise 2026 de l'ASNR concernant ses systèmes d'arrêt d'urgence ?"
query_embedding = model.encode("query: " + question, normalize_embeddings=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2978.95it/s]


In [26]:
all_informations = conn.execute(
    """
    SELECT d.name, c.starting_page, c.ending_page, c.text
    FROM chunk c 
    JOIN document d ON d.id = c.document_id
    ORDER BY c.embedding <=> %s
    LIMIT 5
    """,
    (query_embedding,),
).fetchall()

for name, starting_page, ending_page, text in all_informations:
    (print(name, starting_page, ending_page, text))

dossier_concertation_calogena_2026.pdf 52 52 Calogena a choisi d’engager les discus- sions avec l’ASNR très tôt dans le pro- cessus, depuis début 2024, et a soumis son dossier d’options de sûreté (DOS) en octobre 2024. En mai 2026, l’ASNR a rendu son avis sur la première option de sûreté concernant la démarche de diversification du système d’arrêt de son réacteur. Le dépôt d’une DAC est visé mi-2027.
L’Avant-projet sommaire (APS) s’est achevé en octobre 2025 et l’avant- projet détaillé (APD) a démarré en novembre 2025. Le début de construc- tion est visé fin 2029, pour une mise en service fin 2032.
avis_ASNR_2026.pdf 2 2 L’ASNR considère que CALOGENA devra , dans le cadre d’une éventuelle demande d’autorisation de création, présenter des dispositions de diversifi cation des dispositifs de coupure de l’alimentation électrique des mécanismes de préhension des grappes des deux systèmes d’arrêt d’urgence.
avis_ASNR_2026.pdf 1 1 RÉPUBLIQUE FRANÇAISE

1/4

Avis n° 2026-AV-027 de l’Autorité d

In response to the question, “What must Calogena do following the ASNR’s 2026 expert opinion regarding its emergency shutdown systems?”:        
the model identifies text passages stating that Calogena must propose measures to diversify the power-cut-off devices for the cluster-gripping mechanisms of both emergency shutdown systems. Furthermore, the design options are deemed “satisfactory.” 

The model therefore answered the question correctly by selecting the appropriate text segments. 

## 03. Connection an LLM 

Now that our first model effectively isolates the relevant parts of the documents, we need to connect another model, an LLM, to generate a response. 

Since one of Calogena’s goals is to produce a “sovereign” heating system, we will use a French LLM and choose Mistral. 

So I created a Mistral API key and entered the information in the `.env` file

In [30]:
load_dotenv() 
context = str(all_informations)

system_prompt = """Tu es un assistant IA qui utilise les documents publics disponible sur Calogena pour répondre. 
Répond à cette question en utilisant uniquement les passages fournis. 
Cite le document et la page. 
Si les passages ne permettent pas de répondre à la question, dit que tu ne sais pas et n'invente rien."""

user_prompt = f"""
passages : {context}
Question : {question}
"""

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

chat_response = client.chat.complete(
    model="mistral-medium-latest",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
)

print(chat_response.choices[0].message.content)

D'après les passages fournis, après l'avis d'expertise 2026 de l'ASNR, **Calogena doit présenter des dispositions de diversification des dispositifs de coupure de l’alimentation électrique des mécanismes de préhension des grappes des deux systèmes d’arrêt d’urgence** dans le cadre d’une éventuelle demande d’autorisation de création (DAC).

Source : *avis_ASNR_2026.pdf*, page 2.


The model responds:             
"Based on the provided excerpts, following ASNR Expert Opinion 2026, **Calogena must propose measures to diversify the power-cut-off devices for the cluster-gripping mechanisms of both emergency shutdown systems** as part of a potential application for a construction permit (DAC).       
Source : *avis_ASNR_2026.pdf*, page 2 " 

He therefore answered the question correctly and did not make anything up. 

So what happens if the answer isn't in the document? 

To make it easier to work with, we're going to create a function using the code we just wrote to call the LLM

In [ ]:
def ask_LLM(question : str) -> str:
    """
    Feature for submitting a request to our AI assistant.
    question : str
    """
    query_embedding = model.encode("query: " + question, normalize_embeddings=True)

    all_informations = conn.execute(
        """
        SELECT d.name, c.starting_page, c.ending_page, c.text
        FROM chunk c 
        JOIN document d ON d.id = c.document_id
        ORDER BY c.embedding <=> %s
        LIMIT 5
        """,
        (query_embedding,),
    ).fetchall()

    context = str(all_informations)

    system_prompt = """Tu es un assistant IA qui utilise les documents publics disponible sur Calogena pour répondre. 
    Répond à cette question en utilisant uniquement les passages fournis. 
    Cite le document et la page. 
    Si les passages ne permettent pas de répondre à la question, dit que tu ne sais pas et n'invente rien."""

    user_prompt = f"""
    passages : {context}
    Question : {question}
    """

    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

    chat_response = client.chat.complete(
        model="mistral-medium-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return(chat_response.choices[0].message.content)

In [32]:
ask_LLM("Quelle est la spécialité culinaire d'Aix-en-Provence ?")

"Je ne sais pas. Les passages fournis ne mentionnent pas la spécialité culinaire d'Aix-en-Provence."

When asked, “What is the culinary specialty of Aix-en-Provence?” the model responds:            
“I don't know. The provided passages do not mention the culinary specialty of Aix-en-Provence.”

The model does not make things up when the information is not in the document. It is therefore usable. 

## 04. API 

I'm now going to create an API to receive questions and return the answers from our model. 

This API doesn't need to be complex; it boils down to a single request. To do this, we'll use FastAPI, which automatically generates documentation. 

So I'm going to create the `api.py` file, which replicates what we've already done here with our function. 

As a reminder, our current function: 
- converts the question into a vector based on the embeddings in our database 
- finds the 5 shortest chunks
- sends these chunks to our LLM 
- returns the LLM’s response, which is based on these chunks 

The API is working well